# 🤖 ADAN Trading Bot — Colab T4 GPU Pipeline

**7 étapes :** GPU check → Setup → Data CCXT → Training PBT → Extract model → Paper trading → Validation

> **Avant de commencer :** `Runtime → Change runtime type → GPU (T4)`

## 0. 🔍 Vérification GPU T4

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('❌ Pas de GPU. Runtime → Change runtime type → GPU (T4)')
print(f'✅ GPU: {result.stdout.strip()}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ PyTorch ne voit pas le GPU. Redémarrer le runtime.')
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ CUDA: {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'✅ Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}')


## 1. 📦 Setup — Clone & Installation

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/Cabrel10/ADAN0.git'
REPO_DIR = '/content/ADAN0'

if os.path.exists(REPO_DIR):
    print('📁 Repo présent, mise à jour...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)
else:
    print('📥 Clonage...')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'✅ Répertoire: {os.getcwd()}')

# Ajouter src/ au PYTHONPATH — évite pip install -e . qui peut échouer sur Colab
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ['PYTHONPATH'] = SRC_DIR + ':' + os.environ.get('PYTHONPATH', '')
print(f'✅ PYTHONPATH: {SRC_DIR}')


In [ ]:
# Installation des packages manquants SANS toucher aux versions Colab
import subprocess, sys

PACKAGES = [
    'stable-baselines3>=2.0.0',
    'ray[tune]>=2.0.0',
    'ccxt>=4.0.0',
    'pandas-ta>=0.3.14b',
    'pyarrow>=10.0.0',
    'gymnasium>=0.26.0',
    'shimmy>=0.2.1',
    'optuna>=3.0.0',
]

print('📦 Installation...')
for pkg in PACKAGES:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    icon = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {icon} {pkg}')
    if r.returncode != 0:
        print(f'     {r.stderr[-300:]}')
print('\n✅ Installation terminée')


In [ ]:
import importlib, os

checks = [('ccxt','ccxt'),('ray','ray'),('stable_baselines3','SB3'),
          ('torch','PyTorch'),('pandas_ta','pandas_ta'),('gymnasium','gymnasium')]

all_ok = True
for mod, name in checks:
    try:
        m = importlib.import_module(mod)
        print(f'  ✅ {name}: {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'  ❌ {name}: {e}'); all_ok = False

try:
    import adan_trading_bot
    print(f'  ✅ adan_trading_bot: OK')
except ImportError as e:
    print(f'  ❌ adan_trading_bot: {e}')
    print(f'     Vérifier PYTHONPATH={os.environ.get("PYTHONPATH","non défini")}')
    all_ok = False

print('\n✅ Tous les imports OK' if all_ok else '\n⚠️  Certains imports ont échoué')


## 2. 📊 Téléchargement des données via CCXT (50 000 candles)

In [ ]:
# generate_colab_dataset.py gère:
#   - Binance public (pas de clé API) → fallback Bybit → Bitget
#   - Calcul des indicateurs (RSI, ATR, EMA, MACD, Bollinger...)
#   - Alignement Master Clock 5m/1h/4h
#   - Sauvegarde en .parquet
import subprocess, sys

# IMPORTANT: config.yaml requiert BTCUSDT + XRPUSDT
SYMBOLS = ['BTCUSDT', 'XRPUSDT']
DATA_DIR = 'data/processed/indicators'

for split, n_candles in [('train', 50000), ('test', 10000)]:
    print(f'\n📥 {split} ({n_candles} candles)...')
    cmd = [sys.executable, 'scripts/generate_colab_dataset.py',
           '--output', DATA_DIR, '--symbols', *SYMBOLS,
           '--split', split, '--candles', str(n_candles)]
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print(f'⚠️  Échec live — fallback synthétique...')
        subprocess.run(cmd + ['--synthetic'], check=True)

print('\n✅ Téléchargement terminé')


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('data/processed/indicators')
all_ok = True

for split in ['train', 'test']:
    print(f'\n📁 {split}/')
    for symbol in ['BTCUSDT', 'XRPUSDT']:
        for tf in ['5m', '1h', '4h']:
            fpath = DATA_DIR / split / symbol / f'{tf}.parquet'
            if fpath.exists():
                df = pd.read_parquet(fpath)
                print(f'  ✅ {symbol}/{tf}.parquet — {len(df)} lignes, {len(df.columns)} cols')
                if len(df) < 100: all_ok = False
            else:
                print(f'  ❌ {fpath} MANQUANT'); all_ok = False

print('\n✅ Données OK' if all_ok else '\n❌ Données manquantes — relancer la cellule précédente')


## 3. 🎯 Entraînement PBT — 4 workers sur T4 GPU

In [ ]:
# 'quick' = 100K steps (~10 min) | 'full' = 1M steps (~2h)
TRAINING_MODE = 'quick'

STEPS          = 100_000 if TRAINING_MODE == 'quick' else 1_000_000
STEPS_PER_ITER = 5_000   if TRAINING_MODE == 'quick' else 10_000
NUM_SAMPLES    = 4   # 4 workers partagent le T4
NUM_CPUS       = 2   # Colab limite à 2 CPUs

print(f'⚙️  Mode: {TRAINING_MODE} | Steps: {STEPS:,} | Workers: {NUM_SAMPLES}')
print(f'   GPU: T4 (0.25 GPU/worker) | CPUs: {NUM_CPUS}')
print(f'   Durée estimée: {"~10 min" if TRAINING_MODE == "quick" else "~2 heures"}')


In [ ]:
import subprocess, sys, os

os.environ['PYTHONPATH'] = '/content/ADAN0/src:' + os.environ.get('PYTHONPATH', '')
subprocess.run(['rm', '-rf', '/tmp/ray'], capture_output=True)

# Essai 1: PBT avec Ray Tune (4 workers T4)
cmd_pbt = [
    sys.executable, 'scripts/train_parallel_agents.py',
    '--config', 'config/config.yaml',
    '--steps', str(STEPS),
    '--steps-per-iter', str(STEPS_PER_ITER),
    '--num-samples', str(NUM_SAMPLES),
    '--num-cpus', str(NUM_CPUS),
    '--no-subproc',
    '--profiles', 'scalper', 'intraday', 'swing', 'position',
]

print('🚀 Tentative PBT (Ray Tune, 4 workers)...')
print(f'   Commande: {" ".join(cmd_pbt[1:])}\n')

# stdout + stderr visibles dans la cellule Colab
r = subprocess.run(cmd_pbt, stdout=None, stderr=None)

if r.returncode == 0:
    print('\n✅ PBT terminé avec succès')
else:
    print(f'\n⚠️  PBT échoué (code {r.returncode}) — fallback train_simple_ppo...')
    print('─' * 60)
    # Essai 2: train_simple_ppo (1 worker, plus stable sur Colab)
    cmd_simple = [
        sys.executable, 'scripts/train_simple_ppo.py',
        '--config', 'config/config.yaml',
        '--steps', str(STEPS),
    ]
    print(f'🔄 Fallback: {" ".join(cmd_simple[1:])}\n')
    r2 = subprocess.run(cmd_simple, stdout=None, stderr=None)
    if r2.returncode == 0:
        print('\n✅ train_simple_ppo terminé')
    else:
        print(f'\n❌ Les deux méthodes ont échoué.')
        print('   → Vérifier les logs ci-dessus pour le détail de l\'erreur')
        print('   → Relancer depuis la cellule 0 (Runtime → Restart and run all)')


## 4. 🏆 Extraction du meilleur modèle

In [ ]:
import subprocess, sys, json
from pathlib import Path

r = subprocess.run([sys.executable, 'scripts/extract_best_model.py'])

PROD_DIR = Path('models/rl_agents/production')
if PROD_DIR.exists():
    for f in sorted(PROD_DIR.iterdir()):
        print(f'  ✅ {f.name} ({f.stat().st_size/1024:.1f} KB)')
    meta_f = PROD_DIR / 'metadata.json'
    if meta_f.exists():
        meta = json.loads(meta_f.read_text())
        print('\n📊 Métadonnées:')
        for k, v in meta.items(): print(f'   {k}: {v}')
else:
    print('⚠️  Aucun modèle extrait — vérifier l\'étape 3')


## 5. �� Paper Trading — Wallet virtuel isolé ($20.50)

- Balance initiale: **$20.50** | Max: **$25.00** (Micro Capital)
- Max positions: **1**
- **ZÉRO ordre réel** — 100% simulé localement
- Logs `[INTENTION]` / `[EXECUTION]` pour audit complet

In [ ]:
import subprocess, sys

print('📈 Paper trading offline (60 secondes)...')
r = subprocess.run(
    [sys.executable, 'scripts/paper_trading_monitor.py', '--offline', '--duration', '60'],
    timeout=120
)
print('\n✅ Terminé' if r.returncode == 0 else f'\n⚠️  Code {r.returncode} (normal si modèle absent)')


## 6. ✅ Validation du lifecycle des trades (12 checks)

In [ ]:
import subprocess, sys
from pathlib import Path

# Chercher le log le plus récent
candidates = list(Path('logs').glob('run_*.log')) + \
             sorted(Path('logs/central').glob('*.log'), reverse=True)[:3] \
             if Path('logs/central').exists() else list(Path('logs').glob('run_*.log'))

log_file = next((f for f in candidates if f.exists() and f.stat().st_size > 1000), None)

if log_file is None:
    print('⚠️  Aucun log trouvé — lancer d\'abord l\'entraînement (étape 3)')
else:
    print(f'📋 Validation sur: {log_file}')
    r = subprocess.run([sys.executable, 'scripts/validate_trade_lifecycle.py',
                        str(log_file), '--run-id', 'Colab-Run'])
    print('\n✅ Validation OK' if r.returncode == 0 else f'\n⚠️  Code {r.returncode}')


## 7. 📦 Résultats & Téléchargement

In [ ]:
import os, json, shutil, torch
from pathlib import Path
from datetime import datetime

PROD_DIR   = Path('models/rl_agents/production')
EXPORT_DIR = Path('/content/adan_export')
EXPORT_DIR.mkdir(exist_ok=True)

for fname in ['model.zip', 'vecnormalize.pkl', 'metadata.json']:
    src = PROD_DIR / fname
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / fname)
        print(f'✅ {fname} ({src.stat().st_size/1024:.1f} KB)')
    else:
        print(f'⚠️  {fname} absent')

report = {'timestamp': datetime.now().isoformat(), 'colab_run': True,
          'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}
(EXPORT_DIR / 'run_report.json').write_text(json.dumps(report, indent=2))

archive = shutil.make_archive('/content/adan_model', 'zip', EXPORT_DIR)
print(f'\n📦 Archive: {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')
print('💾 Files panel (icône dossier) → adan_model.zip')


In [ ]:
try:
    from google.colab import files
    import os
    if os.path.exists('/content/adan_model.zip'):
        files.download('/content/adan_model.zip')
    else:
        print('⚠️  Archive non trouvée — vérifier la cellule précédente')
except ImportError:
    print('ℹ️  Télécharger manuellement: /content/adan_model.zip')
